# H-PPO 主训练入口

该笔记本将逐步整合环境、算法与配置，完成 torchrl + gymnasium 的 H-PPO 训练流程。

In [ ]:
# 在 Jupyter 环境中重复运行时，先完全重置 logger 状态
# 这样可以避免重复添加 handler 导致日志输出多次
import sys
sys.path.insert(0, '.')  # 确保能导入本地模块

from algo.logger import reset_logger
reset_logger()  # 清理所有旧的 handler 和状态

In [ ]:
from dataclasses import replace
from pathlib import Path
from typing import Dict
from copy import deepcopy
import math
import torch
import yaml
import os
import numpy as np
import pandas as pd
from collections import OrderedDict
import importlib
from matplotlib import pyplot as plt

from algo import HPPOConfig, build_hppo_modules, train_hppo
from algo.logger import get_logger, setup_logger
from env import (
    MACSimulatorConfig,
    SatelliteMACEnv,
    SatelliteMACEnvConfig,
    default_simulator_config,
    build_gym_env,
)
from torchrl.envs import ParallelEnv, SerialEnv, TransformedEnv
from torchrl.envs.libs.gym import GymWrapper
from torchrl.envs.transforms import Compose, DoubleToFloat
import matplotlib.font_manager as font_manager
import scienceplots
plt.style.use(['science', 'no-latex'])
plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['font.sans-serif'] = ['Noto Sans SC',]
import algo.hppo
importlib.reload(algo.hppo)
from algo import train_hppo

# 导入评估工具函数
from utils.evaluate import evaluate_trained_agent, _scalar, load_yaml_config


In [ ]:
CONFIG_PATH = Path("conf/default.yaml")
raw_config = load_yaml_config(CONFIG_PATH)
seed = int(raw_config.get("seed", 20251202))
torch.manual_seed(seed)
log_section = raw_config.get("logging", {})
log_file = f"{seed}test.log"

log_level = log_section.get("level", "INFO")
setup_logger(log_file, log_level)
my_logger = get_logger()
my_logger.info(f"Logger initialized (level={log_level}, file={log_file or '<stderr>'})")
env_section = dict(raw_config.get("environment", {}))
algo_section = raw_config.get("algorithm", {})
env_section.setdefault("noise_scale", 0.01)
if "motion_per_step" not in env_section:
    env_section["motion_per_step"] = default_simulator_config().motion_per_step


def _override_sim_config(overrides: Dict) -> MACSimulatorConfig:
    base_config = default_simulator_config()
    total = overrides.get("total_preambles")
    split = overrides.get("base_preamble_split")
    if total is not None:
        base_config = replace(base_config, total_preambles=int(total))
    if split is not None:
        if len(split) != 3:
            raise ValueError("base_preamble_split must contain three integers")
        base_config = replace(
            base_config,
            base_preamble_split=tuple(int(x) for x in split),
        )
    noise_scale = overrides.get("noise_scale")
    if noise_scale is not None:
        base_config = replace(base_config, noise_scale=float(noise_scale))
    motion_per_step = overrides.get("motion_per_step")
    if motion_per_step is not None:
        base_config = replace(base_config, motion_per_step=float(motion_per_step))
    # 配置历史长度 (Simulator 层的 history_size)
    # 注意: 由于 HISTORY_DIM 是硬编码的，当前必须保持为 10
    # 如果需要修改，需要同步更新 mac_simulator.py 中的限制
    history_size = overrides.get("history_size", 10)
    if history_size != 10:
        my_logger.warning(
            f"history_size={history_size} requested, but current implementation "
            f"requires history_size=10. Using default value."
        )
        history_size = 10
    base_config = replace(base_config, history_size=int(history_size))
    return base_config


simulator_config = _override_sim_config(env_section)
num_slots_per_step = int(env_section.get("num_slots_per_step", 10))
env_config = SatelliteMACEnvConfig(
    num_slots_per_step=10,
    decision_horizon=100,
    preamble_delta_range=int(env_section.get("preamble_delta_range", 2)),
    flatten_observation=bool(env_section.get("flatten_observation", True)),
    simulator_config=simulator_config,
)
my_logger.info(f"env_config {env_config}")
my_logger.info(f"Simulator history_size={simulator_config.history_size} (history_stats dim={simulator_config.history_size * 6})")
num_envs = 8  # max(1, int(env_section.get("num_envs", 4)))
env_backend = str(
    env_section.get("env_backend", env_section.get("backend", "parallel"))
).lower()
parallel_collection = bool(env_section.get("parallel_collection", True))
if env_backend not in {"serial", "parallel"}:
    my_logger.warning(f"Unknown env_backend={env_backend}, falling back to 'serial'")
    env_backend = "serial"
if parallel_collection:
    env_backend = "parallel"
train_config = HPPOConfig(
    frames_per_batch=3200,  # int(algo_section.get("frames_per_batch", 1024)),
    mini_batch_size=256,  # int(algo_section.get("mini_batch_size", 128)),
    rollout_epochs=4,  # int(algo_section.get("rollbase_preamble_splitout_epochs", 4)),
    max_iterations=1500,  # int(algo_section.get("max_iterations", 100)),
    gamma=float(algo_section.get("gamma", 0.99)),
    gae_lambda=float(algo_section.get("gae_lambda", 0.95)),
    clip_epsilon=0.2,  # float(algo_section.get("clip_epsilon", 0.2)),
    entropy_coeff=0.1,  # float(algo_section.get("entropy_coeff", 0.01)),
    actor_lr=3e-4,  # float(algo_section.get("actor_lr", 3e-4)),
    critic_lr=3e-4,  # float(algo_section.get("critic_lr", 3e-4)),
    device=torch.device(algo_section.get("device", "cpu")),
)
my_logger.info(f"train_config {train_config}")
if train_config.frames_per_batch % num_envs != 0:
    my_logger.warning(
        f"frames_per_batch ({train_config.frames_per_batch}) is not divisible by num_envs ({num_envs}); collector will pad the last mini-batch.",
    )
my_logger.info(
    f"Config ready | seed={seed} frames_per_batch={train_config.frames_per_batch} horizon={env_config.decision_horizon} num_envs={num_envs} backend={env_backend} parallel={parallel_collection}",
)


2025-12-03 19:27:24 | INFO     | Logger initialized (level=INFO, file=42test.log)


Loaded configuration from conf\default.yaml


2025-12-03 19:27:24 | INFO     | env_config SatelliteMACEnvConfig(num_slots_per_step=10, decision_horizon=81, simulator_config=MACSimulatorConfig(regions=(RegionTrafficProfile(name='suburban', cbra_density=1.0, pbra_density=3.5, cfra_density=0.5), RegionTrafficProfile(name='periurbanType1', cbra_density=2.0, pbra_density=2.0, cfra_density=6.0), RegionTrafficProfile(name='periurbanType2', cbra_density=2.0, pbra_density=7.0, cfra_density=1.0), RegionTrafficProfile(name='urban', cbra_density=56.0, pbra_density=16.0, cfra_density=8.0)), segments=(RegionSegment(region_name='suburban', length=2.0), RegionSegment(region_name='periurbanType1', length=1.0), RegionSegment(region_name='urban', length=5.0), RegionSegment(region_name='periurbanType2', length=1.0), RegionSegment(region_name='suburban', length=2.0)), total_preambles=64, base_preamble_split=(27, 27, 10), history_size=10, coverage_window=3.0, motion_per_step=0.1, slots_per_motion=10, noise_scale=0.01, reward_weights={'throughput': 1.0,

## 构建 TorchRL 环境与模块
- 使用 `GymWrapper` + `TransformedEnv` 适配 `SatelliteMACEnv`
- 通过 `build_hppo_modules` 生成混合动作策略与价值网络

In [ ]:
def _make_wrapped_env(rank: int = 0, **_: object) -> GymWrapper:
    env_seed = seed + rank if seed is not None else None
    wrapped = build_gym_env(config=deepcopy(env_config))
    wrapped.set_seed(env_seed)
    # 自动注册info字典 - 会运行一次rollout来收集info specs
    wrapped.auto_register_info_dict()
    return wrapped


In [ ]:
base_env = _make_wrapped_env(0)
td = base_env.reset()
done = False
episode_reward = 0.0
episode_length = 0
while not done:
    td_action = base_env.rand_action()
    td_next = base_env.step(td_action)
    reward = td_next.get(("next", "reward")).mean()
    done = td_next.get(("next", "done"))[0]
    episode_reward += reward
    episode_length += 1
    td = base_env.reset() if done else td_next["next"]



SatelliteMACEnvConfig(num_slots_per_step=10, decision_horizon=81, simulator_config=MACSimulatorConfig(regions=(RegionTrafficProfile(name='suburban', cbra_density=1.0, pbra_density=3.5, cfra_density=0.5), RegionTrafficProfile(name='periurbanType1', cbra_density=2.0, pbra_density=2.0, cfra_density=6.0), RegionTrafficProfile(name='periurbanType2', cbra_density=2.0, pbra_density=7.0, cfra_density=1.0), RegionTrafficProfile(name='urban', cbra_density=56.0, pbra_density=16.0, cfra_density=8.0)), segments=(RegionSegment(region_name='suburban', length=2.0), RegionSegment(region_name='periurbanType1', length=1.0), RegionSegment(region_name='urban', length=5.0), RegionSegment(region_name='periurbanType2', length=1.0), RegionSegment(region_name='suburban', length=2.0)), total_preambles=64, base_preamble_split=(27, 27, 10), history_size=10, coverage_window=3.0, motion_per_step=0.1, slots_per_motion=10, noise_scale=0.01, reward_weights={'throughput': 1.0, 'collision': -0.3}, backoff_strategy=Backof